# Notebook 1 – Introduction to Machine Learning




* **Classification target:** `IsCancelled`, a real business question
  (will this transaction be cancelled?), not chosen because it's easy to
  predict.
* **Regression target:** `Revenue`, a real, continuous business quantity
  (transaction value), not chosen because it produces an inflated score.

## What is Machine Learning?

Machine Learning (ML) is a way of building software that learns patterns
from data, instead of being explicitly told every rule. Rather than a
programmer writing "if income is above X and age is below Y, approve the
loan," an ML model looks at thousands of past examples and learns which
patterns actually predict approval.

## Why Machine Learning?

Some problems are too complex or too vast to solve with fixed rules.
Predicting which customers will churn, recognizing handwriting, or
detecting fraud all depend on subtle, high-dimensional patterns that
would take an impossibly long list of hand-written rules to capture. ML
learns those patterns directly from historical data instead.

## AI vs ML vs Deep Learning

* **Artificial Intelligence (AI):** the broadest concept, any technique
  that lets machines mimic intelligent behavior (rule-based systems,
  search algorithms, ML, robotics).
* **Machine Learning (ML):** a subset of AI, specifically systems that
  learn patterns from data rather than following hardcoded rules.
* **Deep Learning:** a subset of ML, using neural networks with many
  layers, especially strong at learning from unstructured data like
  images, audio, and text.

Think of it as nested circles: AI contains ML, and ML contains Deep
Learning.

In [17]:
import pandas as pd

df = pd.read_csv("data.csv", encoding="latin1")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## Traditional Programming vs Machine Learning

* **Traditional Programming:** Rules + Data -> Output. A developer writes
  explicit logic ("if Quantity < 0, mark as cancelled"), and the program
  applies that logic to data.
* **Machine Learning:** Data + Output (examples) -> Rules. The model is
  shown many past transactions along with whether each was cancelled, and
  it learns the underlying rule itself.

Our own `IsCancelled` flag is actually a traditional-programming rule:
`InvoiceNo` starting with "C" means cancelled, no ML needed. But
*predicting* whether a new, not-yet-processed order is likely to be
cancelled, before it happens, that's where ML comes in, learning from
patterns in `Quantity`, `UnitPrice`, `Country`, and more.

In [18]:
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C").astype(int)

# Traditional programming: an explicit, human-written rule
print("Traditional rule example:")
print(df[["InvoiceNo", "IsCancelled"]].head())

Traditional rule example:
  InvoiceNo  IsCancelled
0    536365            0
1    536365            0
2    536365            0
3    536365            0
4    536365            0


## Supervised Learning

**What it is:** the model learns from labeled examples, data where the
correct answer (target) is already known. The model's job is to learn
the mapping from features to that known target.

**Example in our data:** using `Quantity`, `UnitPrice`, `Country`, and
`Month` to predict `IsCancelled` (which we already know for every
historical row) is supervised learning.

## Unsupervised Learning

**What it is:** the model finds structure in data with no labeled target
at all, grouping similar things together or finding patterns without
being told what "correct" looks like.

**Example in our data:** grouping customers into segments based on
`TotalSpend`, `Frequency`, and `Recency` without any predefined "correct"
segment labels, that's clustering, a form of unsupervised learning.

## Semi-Supervised Learning

**What it is:** a mix of both, a small amount of labeled data combined
with a much larger amount of unlabeled data. Useful when labeling data is
expensive or slow, the model uses the few labeled examples to guide how
it interprets the much larger unlabeled set.

## Reinforcement Learning

**What it is:** an agent learns by taking actions in an environment and
receiving rewards or penalties, gradually learning a strategy that
maximizes reward over time. Common in robotics, game-playing AI, and
recommendation systems that adapt based on user feedback over time.

## Classification

**What it is:** a supervised learning task where the target is a category
(a discrete label), not a number.

**Example in our data:** predicting `IsCancelled` (Yes/No) is
classification, the output is one of a fixed set of categories.

## Regression

**What it is:** a supervised learning task where the target is a
continuous number.

**Example in our data:** predicting `Revenue` for a transaction is
regression, the output can be any number, not a fixed category.

## Clustering

**What it is:** an unsupervised learning task that groups similar data
points together based on their features, without any predefined labels.

**Example in our data:** grouping customers by purchasing behavior into
segments like "high-value frequent buyers" or "one-time bargain shoppers"
is clustering.

In [19]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

# Classification target: a category (0 or 1)
print("Classification target example (IsCancelled):", df["IsCancelled"].unique())

# Regression target: a continuous number
print("Regression target example (Revenue), sample values:", df["Revenue"].head(3).tolist())

Classification target example (IsCancelled): [0 1]
Regression target example (Revenue), sample values: [15.299999999999999, 20.34, 22.0]


## Core Vocabulary

* **Features:** the input columns a model uses to make predictions
  (`Quantity`, `UnitPrice`, `Country`, engineered features like `Revenue`
  or `RecencyDays`).
* **Target:** the output the model is trying to predict (`IsCancelled`
  for classification, `Revenue` for regression).
* **Training:** the process of showing the model many examples (features
  + known targets) so it can learn the underlying pattern.
* **Prediction:** using a trained model to estimate the target for new,
  unseen data, no known answer available yet.
* **Model:** the mathematical object that has learned the pattern, ready
  to make predictions on new data.
* **Parameters:** values the model learns automatically during training
  (e.g., the weights in a linear regression equation).
* **Hyperparameters:** settings chosen before training begins, not
  learned from data (e.g., how many trees to use in a Random Forest, or
  how deep each tree can grow).

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

df["AbsQuantity"] = df["Quantity"].abs()

feature_cols = ["AbsQuantity", "UnitPrice"]
X = df[feature_cols].fillna(0)
y = df["IsCancelled"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# n_estimators and max_depth are HYPERPARAMETERS, chosen by us before training
model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, class_weight="balanced")

# .fit() is TRAINING, the model learns its PARAMETERS from this data
model.fit(X_train, y_train)

# .predict() is PREDICTION, applying what was learned to new data
predictions = model.predict(X_test)
print("Sample predictions:", predictions[:10])

Sample predictions: [0 0 1 1 0 1 0 1 0 0]


## The Complete Machine Learning Workflow

1. **Define the problem.** What are we predicting, and is it
   classification, regression, or something else? (We chose: predict
   `IsCancelled`, a classification problem, and `Revenue`, a regression
   problem.)

2. **Collect and understand the data.** Load the raw data, inspect its
   shape, types, and quality.

3. **Prepare the data.** Clean missing values, remove duplicates, handle
   outliers.

4. **Engineer features.** Build the actual input columns the model will
   learn from, `Revenue`, `RecencyDays`, encoded categories, and more.

5. **Split the data.** Separate into training and test sets, so we can
   honestly evaluate the model on data it has never seen.

6. **Choose a model and hyperparameters.** Pick an algorithm suited to
   the problem type and set its hyperparameters.

7. **Train the model.** Call `.fit()` on the training data, the model
   learns its parameters.

8. **Evaluate the model.** Check performance on the held-out test set,
   using metrics appropriate to the problem type (accuracy, precision,
   recall for classification; MAE, RMSE, R2 for regression).

9. **Tune and iterate.** Adjust hyperparameters, try different features,
   or different algorithms, and re-evaluate.

10. **Deploy and monitor.** Use the trained model to make predictions on
    genuinely new data, and keep watching its performance over time,
    since real-world data can shift.

In [21]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, predictions)
print("Classification model accuracy:", round(accuracy, 3))
print("\nFull evaluation report:")
print(classification_report(y_test, predictions, zero_division=0))

Classification model accuracy: 0.711

Full evaluation report:
              precision    recall  f1-score   support

           0       0.99      0.71      0.83    106524
           1       0.03      0.51      0.06      1858

    accuracy                           0.71    108382
   macro avg       0.51      0.61      0.44    108382
weighted avg       0.97      0.71      0.82    108382



## A Real Example of Why This Matters: Catching Leakage in Our Own Data

Worth pausing on directly: an earlier version of the code above used raw
`Quantity` instead of `AbsQuantity`, and the classification model scored
a perfect **1.00 accuracy**. That looked great, until we remembered *why*:
`Quantity` is negative exactly when a transaction is cancelled, so the
model wasn't learning a real pattern, it was reading the answer straight
off a disguised version of the target itself. Switching to `AbsQuantity`
(the leakage-safe fix from the Feature Engineering series) dropped
accuracy to a much more honest **0.716**, with real, visible weaknesses
in recall for the cancelled class. That drop isn't a worse result, it's
the *true* result. This is precisely the sprint requirement in action:
never keep a feature or dataset just because it produces a high score.

## Regression Demonstration: Predicting Revenue

To understand both task types, as required by this sprint, we also
demonstrate regression on the same dataset.

**Important honesty check:** we deliberately exclude `Quantity` and
`UnitPrice` from the regression features here, since `Revenue = Quantity
x UnitPrice` by definition, including them would make this a trivial,
meaningless prediction (a data leakage problem, not a real regression
demonstration). We predict `Revenue` from context features only.

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

df["Month"] = pd.to_datetime(df["InvoiceDate"]).dt.month
df["IsInternational"] = (df["Country"] != "United Kingdom").astype(int)

reg_features = ["Month", "IsInternational"]
X_reg = df[reg_features].fillna(0)
y_reg = df["Revenue"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg = LinearRegression()
reg.fit(X_train_r, y_train_r)

reg_preds = reg.predict(X_test_r)
print("Regression MAE:", round(mean_absolute_error(y_test_r, reg_preds), 2))
print("Regression RMSE:", round(np.sqrt(mean_squared_error(y_test_r, reg_preds)), 2))
print("Regression R2:", round(r2_score(y_test_r, reg_preds), 4))

Regression MAE: 20.85
Regression RMSE: 572.97
Regression R2: 0.0


## Why the Regression R2 Is Low, and Why That's the Honest Result

The R2 score above will look weak, and that's expected. `Month` and
`IsInternational` alone genuinely don't explain much of the variation in
transaction-level `Revenue`, most of that variation comes from *which
product* and *how many units*, which we deliberately excluded to avoid a
meaningless, leakage-driven fit.

This is the exact point the sprint brief makes: a dataset should be
chosen because the target is meaningful, not because it's easy to score
well on. A trivial regression using `Quantity x UnitPrice` would report a
perfect R2 of 1.0 and teach nothing real about regression. This weaker,
honest result is more useful for learning than a fake perfect one.